In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import libpysal.weights as weights
import esda

# Configuración de visualización
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.precision', 3)

C:\Users\tomas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

## 6. Taller Práctico: Precios de Vivienda en San Diego

Vamos a replicar conceptualmente el **segundo ejemplo empírico** del paper de Getis y Ord: el análisis de precios medios de vivienda en el área metropolitana de San Diego, California, agregados por código postal (ZIP code)

### Motivación del Análisis

En su análisis preliminar, Getis y Ord calcularon los estadísticos globales $G(d)$ e $I(d)$ para estos datos. Los resultados fueron:

- **$I(d)$ global**: Positivo (indicando autocorrelación positiva general)
- **$G(d)$ global**: Negativo (indicando predominancia de valores bajos)

Sin embargo, estos estadísticos globales **no revelaron claramente** la estructura espacial subyacente. ¿Dónde están los clusters? ¿Hay zonas de precios altos y zonas de precios bajos?

### Objetivo

El objetivo de aplicar el estadístico local $G_i^*(d)$ es identificar los **"bolsones" (pockets)** de precios altos (hotspots) y precios bajos (coldspots) que los estadísticos globales no pueden mostrar

### Datos

Trabajaremos con los datos del **Apéndice** del paper, que incluye:
- **Códigos postales** de San Diego
- **Coordenadas** (X, Y) en millas desde un origen arbitrario
- **Precio medio de vivienda** en miles de USD (1990)

¡Comencemos con la implementación!

### 6.1. Carga y Preparación de Datos

### Caso de Estudio: Precios de Vivienda en San Diego (Datos Reales)

Replicaremos el análisis del paper de Getis y Ord usando los **datos reales del Apéndice**. Los datos incluyen 24 códigos postales del área metropolitana de San Diego con:
- **Coordenadas** (X, Y) en millas desde un origen arbitrario
- **Precio medio de vivienda** en miles de USD (1990)

El análisis original identificó hotspots costeros (precios altos) y coldspots centrales (precios bajos) usando un umbral de distancia $d=5$ millas.

In [ ]:
# Datos extraídos del Apéndice (p. 17) del paper
data = {
    'zip_code': ['92024', '92007', '92075', '92014', '92127', '92129', '92128', '92064',
                 '92037', '92122', '92117', '92109', '92110', '92111', '92123', '92108',
                 '92103', '92104', '92105', '92113', '92102', '92107', '92106', '92118'],
    'neighborhood': ['Encinitas', 'Cardiff', 'Solana Beach', 'Del Mar', 'Lake Hodges', 'R. Penasquitos', 'R. Bernardo', 'Poway',
                     'La Jolla', 'University City', 'Clairemont', 'Beaches', 'Bay Park', 'Kearny Mesa', 'Mission Village', 'Mission Valley',
                     'Hillcrest', 'North Park', 'East San Diego', 'Logan Heights', 'East San Diego', 'Ocean Beach', 'Point Loma', 'Coronado'],
    'x': [1, 2, 3, 5, 10, 12, 15, 17, 3, 6, 6, 4, 6, 8, 10, 9, 8, 11, 13, 11, 12, 3, 3, 7],
    'y': [39, 36, 34, 32, 34, 32, 35, 32, 22, 23, 20, 18, 15, 19, 19, 16, 14, 14, 14, 10, 12, 14, 12, 10],
    'price': [264, 260, 261, 309, 265, 194, 191, 236, 398, 201, 192, 249, 152, 138, 131, 89, 225, 152, 111, 84, 88, 229, 338, 374]
}
df = pd.DataFrame(data)

# Convertir a GeoDataFrame usando las coordenadas en millas
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.x, df.y)
)

# Visualizar los primeros registros
print("Primeros registros del dataset:")
print(gdf[['neighborhood', 'x', 'y', 'price']].head(10))
print(f"\nTotal de códigos postales: {len(gdf)}")
print(f"Precio promedio: ${gdf['price'].mean():.1f}k USD")
print(f"Rango de precios: ${gdf['price'].min():.0f}k - ${gdf['price'].max():.0f}k USD")

In [ ]:
# Visualizar los datos base (similar a la Figura 3 del paper [430])
fig, ax = plt.subplots(figsize=(10, 12))
gdf.plot(column='price', ax=ax, legend=True, cmap='viridis', s=100,
         legend_kwds={'label': "Precio de Vivienda (Miles USD)", 'orientation': "horizontal"})

# Anotar algunos puntos clave para referencia
for x, y, label, price in zip(gdf.geometry.x, gdf.geometry.y, gdf.neighborhood, gdf.price):
    if label in ['La Jolla', 'Point Loma', 'Mission Valley', 'East San Diego', 'Coronado', 'Del Mar']:
        ax.annotate(f'{label}\n${price}k', (x, y), textcoords="offset points", 
                    xytext=(3, 3), ha='left', fontsize=8, 
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

ax.set_title('Precios de Vivienda en San Diego por Código Postal [473]', fontsize=14, fontweight='bold')
plt.xlabel('Coordenada X (millas)', fontsize=11)
plt.ylabel('Coordenada Y (millas)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.2. Creación de Pesos Espaciales por Distancia

### Matriz de Pesos Basada en Distancia

Los estadísticos $G_i$ requieren una matriz de pesos espaciales $w_{ij}(d)$ basada en distancia. En el paper, el análisis de $G_i^*$ para San Diego se realizó con un **umbral de distancia** $d=5$ **millas**.

Vamos a replicar esto usando `libpysal.weights.DistanceBand`, que crea una matriz binaria donde:

$$w_{ij} = \begin{cases} 1 & \text{si } \text{distancia}(i,j) \leq d \\ 0 & \text{si } \text{distancia}(i,j) > d \end{cases}$$

Esta matriz define qué códigos postales se consideran "vecinos" de cada código postal focal.

In [ ]:
# Definir el umbral de distancia (d) en 5 millas, como en el paper
d_threshold = 5.0

# Crear la matriz de pesos basada en banda de distancia
# 'binary=True' es el default y replica la matriz {0,1} del paper
W = weights.DistanceBand.from_dataframe(gdf, d_threshold, binary=True)

# Es importante que W no esté estandarizada por fila para este estadístico
W.transform = 'B'  # 'B' for binary (sin estandarizar)

print(f"Matriz de pesos creada con umbral d = {d_threshold} millas")
print(f"Número de ubicaciones: {W.n}")
print(f"Promedio de vecinos por ubicación: {W.mean_neighbors:.2f}")
print(f"Rango de vecinos: {W.min_neighbors} - {W.max_neighbors}")

# Inspeccionemos los vecinos de 'Point Loma' (ZIP 92106)
point_loma_idx = gdf[gdf['neighborhood'] == 'Point Loma'].index[0]
neighbors_idx = W.neighbors[point_loma_idx]

print(f"\nVecinos de 'Point Loma' (a {d_threshold} millas o menos):")
if len(neighbors_idx) > 0:
    print(gdf.loc[neighbors_idx][['neighborhood', 'price']])
else:
    print("No tiene vecinos dentro del umbral.")

### 6.3. Cálculo de $G_i^*$ con `esda`

Ahora aplicaremos el estadístico $G_i^*$ usando la función `esda.G_Local`. 

**Parámetros clave:**
- `y`: La variable de interés (precios de vivienda)
- `W`: La matriz de pesos espaciales
- `star=True`: **Fundamental** para usar la versión $G_i^*$ (incluyente) como en la Tabla 1 y Figura 4 del paper

La función calculará automáticamente los Z-scores y p-values para cada ubicación.

In [ ]:
# Variable a analizar (debe ser un array de numpy)
y = gdf['price'].values

# Calcular el estadístico G_Local (G_i*)
# star=True indica que usamos la versión G_i* (incluye el valor propio)
g_local_star = esda.G_Local(y, W, star=True)

# Añadir los Z-scores (Zs) y p-values (p_sim) al GeoDataFrame
gdf['G_star'] = g_local_star.Gs  # Valores crudos del estadístico
gdf['G_star_Zs'] = g_local_star.Zs  # Z-scores
gdf['G_star_p_sim'] = g_local_star.p_sim  # p-values basados en simulación (permutaciones)

# Ver los resultados, ordenados por Z-score (de mayor a menor)
print("=" * 80)
print("RESULTADOS DEL ANÁLISIS G_i* (d=5 millas)")
print("=" * 80)
result_df = gdf[['neighborhood', 'price', 'G_star_Zs', 'G_star_p_sim']].copy()
result_df = result_df.sort_values('G_star_Zs', ascending=False)
result_df['significance'] = result_df['G_star_p_sim'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.10 else 'ns'))
)
print(result_df.to_string(index=False))
print("\nSignificancia: *** p<0.01, ** p<0.05, * p<0.10, ns = no significativo")

### 6.4. Visualización e Interpretación (Hotspots y Coldspots)

### Estrategia de Visualización

Vamos a visualizar nuestros resultados replicando la lógica de la **Figura 4** del paper. Crearemos dos mapas complementarios:

1. **Mapa de Z-scores crudos**: Muestra la distribución espacial continua de los valores. Los colores rojos indican Z-scores positivos (tendencia a hotspot) y los azules indican Z-scores negativos (tendencia a coldspot).

2. **Mapa de significancia estadística**: Filtra solo los códigos postales donde el Z-score es extremo ($|Z_i| > 1.96$) **y** el p-value es estadísticamente significativo ($p < 0.05$). Este es el mapa que identifica definitivamente los hotspots y coldspots.

In [ ]:
# Mapa 1: Visualización de los Z-scores
fig, ax = plt.subplots(figsize=(10, 12))

# Usamos un mapa de color divergente (Rojo-Azul)
# 'coolwarm' o 'RdBu_r' son buenas opciones
# Rojo = valores positivos (hotspots), Azul = valores negativos (coldspots)
gdf.plot(
    column='G_star_Zs',
    cmap='coolwarm',
    legend=True,
    ax=ax,
    s=120,
    edgecolor='black',
    linewidth=0.5,
    legend_kwds={'label': "Z-score $G_i^*$ (d=5 millas)", 'shrink': 0.8}
)

# Anotar los valores Z en el mapa
for x, y, label in zip(gdf.geometry.x, gdf.geometry.y, gdf.G_star_Zs):
    ax.annotate(f"{label:.2f}", (x, y), textcoords="offset points", 
                xytext=(0, -8), ha='center', fontsize=7, 
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.6))

ax.set_title('Análisis $G_i^*$ de Precios de Vivienda: Z-scores\n(Rojo = Hotspots, Azul = Coldspots)', 
             fontsize=13, fontweight='bold')
plt.xlabel('Coordenada X (millas)', fontsize=11)
plt.ylabel('Coordenada Y (millas)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()v

### Interpretación del Mapa de Z-scores

En el mapa anterior observamos:

- **Z-scores fuertemente positivos (rojo oscuro)**: Se concentran en las zonas **costeras** del área metropolitana, particularmente en códigos postales como **La Jolla**, **Point Loma**, **Coronado** y **Del Mar**. Estos son indicadores de **hotspots** de precios altos.

- **Z-scores fuertemente negativos (azul oscuro)**: Se observan en zonas más **centrales/orientales** como **Mission Valley**, **East San Diego**, **Logan Heights** y **Kearny Mesa**. Estos son indicadores de **coldspots** de precios bajos.

- **Z-scores cercanos a cero (blanco/amarillo)**: Representan áreas donde el patrón espacial es consistente con aleatoriedad.

Esto coincide exactamente con los hallazgos del paper, que identificó los distritos costeros como positivamente asociados (hotspots) y los distritos centrales/interiores como negativamente asociados (coldspots)<sup>[434, 436, 447]</sup>.

In [ ]:
# Mapa 2: Visualización de Significancia Estadística
# Nivel de significancia
alpha = 0.05
z_critical = 1.96

# Clasificar ubicaciones
gdf['cluster_type'] = 'No Significativo'
gdf.loc[(gdf['G_star_Zs'] > z_critical) & (gdf['G_star_p_sim'] < alpha), 'cluster_type'] = 'Hotspot (HH)'
gdf.loc[(gdf['G_star_Zs'] < -z_critical) & (gdf['G_star_p_sim'] < alpha), 'cluster_type'] = 'Coldspot (LL)'

# Contar clusters
n_hotspots = (gdf['cluster_type'] == 'Hotspot (HH)').sum()
n_coldspots = (gdf['cluster_type'] == 'Coldspot (LL)').sum()
n_ns = (gdf['cluster_type'] == 'No Significativo').sum()

print(f"Clusters identificados (α={alpha}, |Z| > {z_critical}):")
print(f"  - Hotspots (HH): {n_hotspots}")
print(f"  - Coldspots (LL): {n_coldspots}")
print(f"  - No significativos: {n_ns}")

In [ ]:


# Crear figura
fig, ax = plt.subplots(figsize=(10, 12))

# Mapear colores
colors = {'Hotspot (HH)': 'red', 'Coldspot (LL)': 'blue', 'No Significativo': 'lightgrey'}
gdf['color'] = gdf['cluster_type'].map(colors)

# CORRECCIÓN: Usar markersize en lugar de s
gdf.plot(
    color=gdf['color'],
    ax=ax,
    markersize=150,  # Cambiar 's' por 'markersize'
    edgecolor='black',
    linewidth=1.0
)

# Anotar los barrios significativos
for x, y, label, cluster in zip(gdf.geometry.x, gdf.geometry.y, gdf.neighborhood, gdf.cluster_type):
    if cluster != 'No Significativo':
        ax.annotate(label, xy=(x, y), xytext=(3, 3), textcoords="offset points", fontsize=8, color='black')

ax.set_title('Clusters Espaciales de Precios de Vivienda\n(Estadístico G_i* Local)', fontsize=14, weight='bold')
ax.set_xlabel('Coordenada X (millas)', fontsize=12)
ax.set_ylabel('Coordenada Y (millas)', fontsize=12)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', edgecolor='black', label='Hotspot (HH)'),
    Patch(facecolor='blue', edgecolor='black', label='Coldspot (LL)'),
    Patch(facecolor='lightgrey', edgecolor='black', label='No Significativo')
]
ax.legend(handles=legend_elements, loc='upper right', title='Tipo de Cluster')

plt.tight_layout()
plt.show()